In [31]:
#as we know that the llm have limited context window so we can't provide all the previous messages to the llm every time so for that we use short term memory which only keeps the recent messages in context window and rest of the messages are stored in database or file system or any other persistent storage. So here we will see how to implement short term memory using langgraph.
#for this we trim down all the previous messages except the recent n messages, tokens, or chars of the messages 



In [32]:
from langgraph.graph import StateGraph,START,END,MessagesState 
from langgraph.checkpoint.memory import InMemorySaver 
from langgraph.checkpoint.sqlite import SqliteSaver

from langchain_core.messages import BaseMessage,HumanMessage,AIMessage 
from langchain_core.messages.utils import trim_messages,count_tokens_approximately

In [33]:
from dotenv import load_dotenv 
load_dotenv()
import sqlite3

In [34]:
from langchain_groq import ChatGroq
llm=ChatGroq(model='Llama-3.3-70b-Versatile')


In [35]:
conn = sqlite3.connect(database="demo.db", check_same_thread=False)
checkpointer = SqliteSaver(conn=conn)

In [36]:
#lets design the graph for chatting with the llm 

graph=StateGraph(MessagesState)
MAX_TOKENS=500

In [37]:
def chat_with_llm(state: MessagesState)-> MessagesState:
    #here we have to add the code for trimming the messages   
    trimmed_messages=trim_messages(state['messages'],strategy='last',token_counter=count_tokens_approximately,max_tokens=MAX_TOKENS)
    
    
    print('Current Token Count ->', count_tokens_approximately(messages=trimmed_messages))
    for message in trimmed_messages:
        print(message.content)
    
    response=llm.invoke(trimmed_messages) 
    
    return {'messages':[response]}

In [38]:
graph.add_node('chat',chat_with_llm)

graph.add_edge(START,'chat') 
graph.add_edge('chat',END)

In [39]:
workflow=graph.compile(checkpointer=checkpointer)

In [40]:
config={'configurable':{'thread_id':'32'}} 

In [61]:
# initial_state={'messages': [HumanMessage(content="Hi, My name is aashish")]}
# initial_state={'messages': [HumanMessage(content="What is my name")]}
# initial_state={'messages':[HumanMessage(content='can you tell me a joke')]}
initial_state={'messages': [HumanMessage(content="write an paragraph about krishna")]}

In [62]:
final_state=workflow.invoke(input=initial_state,config=config)

Current Token Count -> 364
write an paragraph about Elon Musk
Elon Musk is a visionary entrepreneur and business magnate who has revolutionized multiple industries through his innovative ventures. Born on June 28, 1971, in Pretoria, South Africa, Musk's passion for technology and entrepreneurship was evident from an early age. He co-founded his first company, Zip2, at the age of 24, and later sold it for over $300 million. He then went on to co-found X.com, which later became PayPal, and sold it to eBay for $1.5 billion. However, it was his subsequent ventures that truly showcased his visionary spirit. As the CEO of SpaceX, Musk is working towards making humanity a multi-planetary species by developing reusable rockets and aiming to establish a human settlement on Mars. Meanwhile, as the CEO of Tesla, he has been at the forefront of the electric vehicle revolution, making sustainable energy and transportation accessible to the masses. Through his other ventures, such as Neuralink and T

In [63]:
final_state

{'messages': [HumanMessage(content='Hi, My name is aashish', additional_kwargs={}, response_metadata={}, id='4be32ed6-a9a6-4871-91c1-319772b27d3a'),
  AIMessage(content="Hello Aashish, it's nice to meet you. Is there something I can help you with or would you like to chat?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 28, 'prompt_tokens': 43, 'total_tokens': 71, 'completion_time': 0.045524032, 'prompt_time': 0.007205988, 'queue_time': 0.059724912, 'total_time': 0.05273002}, 'model_name': 'Llama-3.3-70b-Versatile', 'system_fingerprint': 'fp_c06d5113ec', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None}, id='run--59a0b8dc-7094-41be-bbf1-6fc749b3c39a-0', usage_metadata={'input_tokens': 43, 'output_tokens': 28, 'total_tokens': 71}),
  HumanMessage(content='What is my name', additional_kwargs={}, response_metadata={}, id='75bd054f-8b30-403b-bccb-d73070088000'),
  AIMessage(content='Your name is Aashish.', additional_kwargs={}, response

In [60]:
#This is simple short term memory which remains for the current execution only

In [56]:
#if we wants that it doesn't gets deleted after the execution we have to use the dbms persistence